<a href="https://colab.research.google.com/github/nagasivaninandam/Text-to-Image-Generator/blob/master/Simple_Text_to_Image_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# minimal_t2i.py
from diffusers import StableDiffusionPipeline
import torch

model_id = "runwayml/stable-diffusion-v1-5"  # simple, smaller than SDXL

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float32,            # use float32 on CPU
    safety_checker=None,                  # optional to avoid extra overhead
)

pipe = pipe.to("cpu")                     # CPU execution

prompt = "a cozy reading nook by a window, soft morning light, illustration"
image = pipe(prompt, num_inference_steps=25, guidance_scale=7.0, height=512, width=512).images[0]
image.save("output.png")
print("Saved output.png")




In [ ]:
# app.py
import gradio as gr
from diffusers import StableDiffusionPipeline
import torch

pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float32, safety_checker=None)
pipe = pipe.to("cpu")

def generate(prompt, steps, guidance, size):
    w, h = (512, 512)
    if size == "Small (384)": w = h = 384
    elif size == "Medium (448)": w = h = 448
    image = pipe(prompt, num_inference_steps=steps, guidance_scale=guidance, width=w, height=h).images[0]
    return image

demo = gr.Interface(
    fn=generate,
    inputs=[gr.Textbox("A watercolor fox in the forest"), gr.Slider(8, 30, value=18, step=1, label="Steps"),
            gr.Slider(3.0, 12.0, value=7.0, step=0.5, label="Guidance"),
            gr.Radio(["Small (384)", "Medium (448)", "Default (512)"], value="Default (512)", label="Size")],
    outputs="image",
    title="Simple Text-to-Image"
)

demo.launch()
